# 🎯 Surveillance-IA — Pipeline Complet (Détection + Reconnaissance + Tracking)

**Projet SAHELYS** — Détection de personnes + Reconnaissance faciale + Suivi de présence

| Composant | Technologie | Rôle |
|-----------|------------|------|
| Détection | **YOLOv8n pré-entraîné** | Détecter les personnes (aucun entraînement !) |
| Reconnaissance | **InsightFace buffalo_l** | Identifier les visages |
| Tracking | **ByteTrack** | Suivre chaque personne |
| Temps de présence | Timer par ID | Durée de séjour |

**Auteur** : BAWANA Théodore — [theo.portefolio.io](https://theo.portefolio.io)

---

⚠️ **Pourquoi pas de fine-tuning ?**
> YOLOv8 pré-entraîné sur COCO détecte déjà `person` (classe 0) avec **mAP > 95%**.
> Le fine-tuning était inutile — on gagne **des heures** en utilisant directement le modèle.

**Durée totale : ~5 minutes** (au lieu de 2+ heures)


## 1️⃣ Setup — Dépendances & GPU

In [ ]:
# ══════════════════════════════════════════════════════════
#  SETUP : Dépendances + GPU
# ══════════════════════════════════════════════════════════
import os, sys, time, json, shutil, pickle
from pathlib import Path

# ── Installer les dépendances ──
!pip install -q ultralytics>=8.2.0 insightface onnxruntime-gpu opencv-python-headless

import torch
import numpy as np
import cv2

# ── Vérifier GPU ──
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9 if hasattr(torch.cuda.get_device_properties(0), 'total_mem') else torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU : {gpu_name} ({vram_gb:.1f} GB)")
else:
    print("⚠️ Pas de GPU — ça marchera quand même (plus lent)")

print(f"✅ PyTorch : {torch.__version__}")
print(f"✅ Setup terminé en quelques secondes !")

## 2️⃣ YOLOv8n — Détection de personnes (AUCUN entraînement !)

> Le modèle pré-entraîné détecte déjà 80 classes dont `person` (classe 0).
> On filtre juste la classe `person` → **0 seconde de training**.

In [ ]:
# ══════════════════════════════════════════════════════════
#  YOLOV8n — Détection personne (pré-entraîné)
# ══════════════════════════════════════════════════════════
from ultralytics import YOLO

# Charger le modèle nano (le plus rapide, 3.2M params)
model = YOLO("yolov8n.pt")
print(f"✅ YOLOv8n chargé — {sum(p.numel() for p in model.model.parameters())/1e6:.1f}M params")
print(f"   Classe 'person' = index 0 (déjà intégrée)")

# ── Test rapide sur une image web ──
!wget -q -O test_person.jpg "https://ultralytics.com/images/bus.jpg"

results = model.predict("test_person.jpg", conf=0.3, classes=[0])  # Filtre: personnes uniquement
n_persons = len(results[0].boxes)
print(f"\n🔍 Test : {n_persons} personne(s) détectée(s) !")
print(f"   → Le modèle fonctionne parfaitement SANS aucun entraînement !")

# Afficher
from IPython.display import Image, display
annotated = results[0].plot()
cv2.imwrite("test_annotated.jpg", annotated)
display(Image("test_annotated.jpg", width=600))

## 3️⃣ InsightFace buffalo_l — Reconnaissance faciale

**Pipeline de reconnaissance :**
```
Image → YOLO (détection personne) → Crop visage → InsightFace → Embedding 512D → Comparaison base
```

| Modèle | Précision | Vitesse |
|--------|----------|---------|
| **buffalo_l** | 99.83% LFW | ~15ms/visage |


In [ ]:
# ══════════════════════════════════════════════════════════
#  INSIGHTFACE — Chargement du modèle buffalo_l
# ══════════════════════════════════════════════════════════
import insightface
from insightface.app import FaceAnalysis

# Charger buffalo_l (détection + reconnaissance faciale)
face_app = FaceAnalysis(
    name="buffalo_l",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
)
face_app.prepare(ctx_id=0, det_size=(640, 640))

print("✅ InsightFace buffalo_l chargé !")
print("   → Détection de visage + Embedding 512D")
print("   → Précision : 99.83% sur LFW benchmark")

# ── Test rapide ──
test_img = cv2.imread("test_person.jpg")
faces = face_app.get(test_img)
print(f"\n🔍 Test : {len(faces)} visage(s) détecté(s)")
for i, face in enumerate(faces):
    print(f"   Visage {i+1}: score={face.det_score:.2f}, embedding={face.embedding.shape}")

## 4️⃣ Base de Personnes — Enregistrement (Nom, Prénom, Visage)

**Système d'identification :**
1. On enregistre une photo de chaque personne avec son **nom** et **prénom**
2. InsightFace extrait un **embedding 512D** unique du visage
3. À chaque détection, on compare l'embedding avec la base → **identification**

> Seuil de similarité cosinus : **≥ 0.4** = même personne

In [ ]:
# ══════════════════════════════════════════════════════════
#  GESTIONNAIRE DE PERSONNES — Enregistrement & Identification
# ══════════════════════════════════════════════════════════
import numpy as np
from datetime import datetime

class PersonManager:
    """Base de données des personnes connues (nom, prénom, embedding)."""

    def __init__(self, similarity_threshold=0.4):
        self.persons = {}          # {id: {"nom": ..., "prenom": ..., "embedding": ...}}
        self.threshold = similarity_threshold
        self.next_id = 1

    def register(self, nom: str, prenom: str, image, face_app):
        """Enregistre une personne à partir de sa photo."""
        if isinstance(image, str):
            image = cv2.imread(image)

        faces = face_app.get(image)
        if not faces:
            print(f"❌ Aucun visage détecté pour {prenom} {nom}")
            return None

        # Prendre le visage le plus grand (le plus proche)
        face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1]))
        embedding = face.embedding / np.linalg.norm(face.embedding)

        pid = self.next_id
        self.persons[pid] = {
            "nom": nom,
            "prenom": prenom,
            "embedding": embedding,
            "date_enregistrement": datetime.now().isoformat(),
        }
        self.next_id += 1

        print(f"✅ Personne enregistrée : {prenom} {nom} (ID={pid})")
        return pid

    def identify(self, face_embedding):
        """Identifie un visage par comparaison cosinus avec la base."""
        if not self.persons:
            return None, 0.0

        query = face_embedding / np.linalg.norm(face_embedding)
        best_id, best_sim = None, -1

        for pid, data in self.persons.items():
            sim = np.dot(query, data["embedding"])
            if sim > best_sim:
                best_sim = sim
                best_id = pid

        if best_sim >= self.threshold:
            p = self.persons[best_id]
            return f"{p['prenom']} {p['nom']}", best_sim
        return None, best_sim

    def save(self, path):
        """Sauvegarde la base sur disque."""
        data = {}
        for pid, info in self.persons.items():
            data[pid] = {
                "nom": info["nom"],
                "prenom": info["prenom"],
                "embedding": info["embedding"].tolist(),
                "date_enregistrement": info["date_enregistrement"],
            }
        with open(path, "w") as f:
            json.dump(data, f, indent=2)
        print(f"💾 Base sauvegardée : {path} ({len(data)} personnes)")

    def load(self, path):
        """Charge la base depuis le disque."""
        with open(path) as f:
            data = json.load(f)
        for pid_str, info in data.items():
            pid = int(pid_str)
            self.persons[pid] = {
                "nom": info["nom"],
                "prenom": info["prenom"],
                "embedding": np.array(info["embedding"], dtype=np.float32),
                "date_enregistrement": info["date_enregistrement"],
            }
            self.next_id = max(self.next_id, pid + 1)
        print(f"📂 Base chargée : {len(self.persons)} personnes")

    def list_persons(self):
        """Liste toutes les personnes enregistrées."""
        for pid, info in self.persons.items():
            print(f"  ID={pid} : {info['prenom']} {info['nom']} (enr. {info['date_enregistrement'][:10]})")

# ── Créer le gestionnaire ──
person_mgr = PersonManager(similarity_threshold=0.4)
print("✅ PersonManager initialisé")
print("   → Prêt à enregistrer des personnes")

In [ ]:
# ══════════════════════════════════════════════════════════
#  ENREGISTREMENT DES PERSONNES
# ══════════════════════════════════════════════════════════
# Option 1 : Upload de photos depuis votre PC
# Option 2 : Webcam Colab (si disponible)
# ══════════════════════════════════════════════════════════

from google.colab import files as colab_files
from IPython.display import display, HTML

print("📸 ENREGISTREMENT DES PERSONNES")
print("=" * 50)
print("Pour chaque personne :")
print("  1. Uploadez une photo avec le visage bien visible")
print("  2. Entrez le nom et prénom")
print()

# ── Fonction d'enregistrement interactif ──
def register_person_interactive():
    """Upload une photo et enregistre la personne."""
    prenom = input("Prénom : ").strip()
    nom = input("Nom : ").strip()

    if not prenom or not nom:
        print("❌ Nom/prénom requis !")
        return

    print(f"\n📤 Uploadez la photo de {prenom} {nom} :")
    uploaded = colab_files.upload()

    if not uploaded:
        print("❌ Aucune photo uploadée")
        return

    filename = list(uploaded.keys())[0]
    pid = person_mgr.register(nom, prenom, filename, face_app)

    if pid:
        # Afficher le visage détecté
        img = cv2.imread(filename)
        faces = face_app.get(img)
        if faces:
            face = faces[0]
            bbox = face.bbox.astype(int)
            cv2.rectangle(img, (bbox[0], bbox[1]), (bbox[2], bbox[3]), (0, 255, 0), 2)
            cv2.putText(img, f"{prenom} {nom}", (bbox[0], bbox[1]-10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            out_path = f"registered_{pid}.jpg"
            cv2.imwrite(out_path, img)
            from IPython.display import Image
            display(Image(out_path, width=400))

# ── Enregistrer plusieurs personnes ──
n_persons = int(input("Combien de personnes à enregistrer ? "))

for i in range(n_persons):
    print(f"\n{'─'*40}")
    print(f"  Personne {i+1}/{n_persons}")
    print(f"{'─'*40}")
    register_person_interactive()

print(f"\n{'='*50}")
print(f"  📋 PERSONNES ENREGISTRÉES : {len(person_mgr.persons)}")
print(f"{'='*50}")
person_mgr.list_persons()

## 5️⃣ Pipeline Complet — Détection + Reconnaissance + Temps de présence

**Flux complet sur une vidéo :**
```
Frame → YOLOv8n (personne?) → InsightFace (qui?) → Tracking (combien de temps?)
```

| Étape | Action |
|-------|--------|
| **Détection** | YOLOv8n filtre classe `person` |
| **Reconnaissance** | InsightFace compare le visage avec la base |
| **Tracking** | On suit chaque personne identifiée |
| **Temps** | On calcule la durée de présence de chaque personne |

In [ ]:
# ══════════════════════════════════════════════════════════
#  TRACKER DE PRÉSENCE — Temps de séjour par personne
# ══════════════════════════════════════════════════════════
from collections import defaultdict
from datetime import datetime, timedelta

class PresenceTracker:
    """Suit le temps de présence de chaque personne identifiée."""

    def __init__(self):
        self.active = {}       # {name: {"first_seen": ts, "last_seen": ts, "frames": n}}
        self.history = []      # Liste des passages terminés
        self.unknown_count = 0

    def update(self, name: str, timestamp: float):
        """Met à jour la présence d'une personne."""
        if name not in self.active:
            self.active[name] = {
                "first_seen": timestamp,
                "last_seen": timestamp,
                "frames": 1,
            }
        else:
            self.active[name]["last_seen"] = timestamp
            self.active[name]["frames"] += 1

    def get_duration(self, name: str) -> float:
        """Retourne la durée de présence en secondes."""
        if name in self.active:
            return self.active[name]["last_seen"] - self.active[name]["first_seen"]
        return 0.0

    def format_duration(self, seconds: float) -> str:
        """Formate la durée en HH:MM:SS."""
        h = int(seconds // 3600)
        m = int((seconds % 3600) // 60)
        s = int(seconds % 60)
        if h > 0:
            return f"{h}h{m:02d}m{s:02d}s"
        elif m > 0:
            return f"{m}m{s:02d}s"
        else:
            return f"{s}s"

    def finalize(self, timeout=5.0, current_time=None):
        """Finalise les personnes parties (pas vues depuis timeout secondes)."""
        if current_time is None:
            current_time = time.time()

        to_remove = []
        for name, data in self.active.items():
            if current_time - data["last_seen"] > timeout:
                duration = data["last_seen"] - data["first_seen"]
                self.history.append({
                    "name": name,
                    "duration": duration,
                    "frames": data["frames"],
                    "arrived": datetime.fromtimestamp(data["first_seen"]).strftime("%H:%M:%S"),
                    "left": datetime.fromtimestamp(data["last_seen"]).strftime("%H:%M:%S"),
                })
                to_remove.append(name)

        for name in to_remove:
            del self.active[name]

    def get_report(self) -> str:
        """Génère un rapport de présence."""
        lines = ["═" * 55]
        lines.append(f"  {'RAPPORT DE PRÉSENCE':^51}")
        lines.append("═" * 55)

        # Personnes encore présentes
        if self.active:
            lines.append("\n  🟢 PRÉSENTS :")
            for name, data in self.active.items():
                dur = self.format_duration(data["last_seen"] - data["first_seen"])
                lines.append(f"     {name:.<30} {dur}")

        # Historique
        if self.history:
            lines.append("\n  📋 HISTORIQUE :")
            for entry in self.history:
                dur = self.format_duration(entry["duration"])
                lines.append(f"     {entry['name']:.<25} {entry['arrived']} → {entry['left']} ({dur})")

        lines.append("═" * 55)
        return "\n".join(lines)


tracker = PresenceTracker()
print("✅ PresenceTracker initialisé")
print("   → Suit le temps de présence de chaque personne identifiée")

## 6️⃣ Traitement Vidéo — Pipeline en action

In [ ]:
# ══════════════════════════════════════════════════════════
#  PIPELINE COMPLET — Vidéo / Webcam
# ══════════════════════════════════════════════════════════

def process_frame(frame, model, face_app, person_mgr, tracker, timestamp):
    """Traite une frame : détection + reconnaissance + tracking."""

    # 1. Détection de personnes avec YOLO
    results = model.predict(frame, conf=0.3, classes=[0], verbose=False)[0]
    annotated = frame.copy()

    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])

        # 2. Extraire la zone de la personne pour reconnaissance faciale
        person_crop = frame[max(0,y1):y2, max(0,x1):x2]
        if person_crop.size == 0:
            continue

        faces = face_app.get(person_crop)
        name = "Inconnu"
        similarity = 0.0

        if faces:
            # 3. Identifier le visage
            face = max(faces, key=lambda f: f.det_score)
            name, similarity = person_mgr.identify(face.embedding)
            if name is None:
                name = "Inconnu"
                tracker.unknown_count += 1

        # 4. Tracker la présence
        if name != "Inconnu":
            tracker.update(name, timestamp)

        # 5. Dessiner les annotations
        duration = tracker.get_duration(name) if name != "Inconnu" else 0
        dur_str = tracker.format_duration(duration) if duration > 0 else ""

        color = (0, 255, 0) if name != "Inconnu" else (0, 0, 255)
        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)

        label = f"{name}"
        if similarity > 0:
            label += f" ({similarity:.0%})"
        if dur_str:
            label += f" | {dur_str}"

        # Fond pour le texte
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(annotated, (x1, y1-th-10), (x1+tw, y1), color, -1)
        cv2.putText(annotated, label, (x1, y1-5),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

    return annotated


def process_video(video_path, output_path="output_surveillance.mp4",
                  model=None, face_app=None, person_mgr=None,
                  tracker_override=None, max_frames=300):
    """Traite une vidéo complète avec le pipeline.

    Args:
        video_path: Chemin de la vidéo source
        output_path: Chemin de la vidéo annotée en sortie
        model: Modèle YOLO (utilise le global si None)
        face_app: InsightFace app (utilise le global si None)
        person_mgr: PersonManager (utilise le global si None)
        tracker_override: PresenceTracker custom (crée un nouveau si None)
        max_frames: Nombre max de frames à traiter
    """
    # Utiliser les globales si non spécifié
    _model = model or globals().get('model')
    _face_app = face_app or globals().get('face_app')
    _person_mgr = person_mgr or globals().get('person_mgr')
    _tracker = tracker_override or PresenceTracker()

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    frame_idx = 0
    t0 = time.time()
    timestamp = 0

    while cap.isOpened() and frame_idx < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        timestamp = frame_idx / fps
        annotated = process_frame(frame, _model, _face_app, _person_mgr, _tracker, timestamp)
        writer.write(annotated)

        frame_idx += 1
        if frame_idx % 50 == 0:
            elapsed = time.time() - t0
            fps_actual = frame_idx / elapsed
            print(f"  Frame {frame_idx}/{min(total, max_frames)} | {fps_actual:.1f} FPS")

    cap.release()
    writer.release()

    # Finaliser tous les trackings
    _tracker.finalize(timeout=0, current_time=timestamp + 10)

    elapsed = time.time() - t0
    print(f"\n✅ Vidéo traitée en {elapsed:.1f}s ({frame_idx} frames)")
    print(_tracker.get_report())
    return _tracker

print("✅ Pipeline prêt !")
print("   → Uploadez une vidéo ou utilisez la webcam")

In [ ]:
# ══════════════════════════════════════════════════════════
#  OPTION 1 : Traiter une vidéo uploadée
# ══════════════════════════════════════════════════════════
from google.colab import files as colab_files

print("📤 Uploadez une vidéo de test (mp4, avi...) :")
print("   (ou passez à la cellule suivante pour le mode webcam)\n")

try:
    uploaded = colab_files.upload()
    if uploaded:
        video_file = list(uploaded.keys())[0]
        print(f"\n🎬 Traitement de : {video_file}")
        result_tracker = process_video(
            video_path=video_file,
            output_path="output_surveillance.mp4",
            max_frames=500
        )

        # Télécharger le résultat
        colab_files.download("output_surveillance.mp4")
except Exception as e:
    print(f"⏭️ Pas de vidéo uploadée — passez à la cellule suivante")

In [ ]:
# ══════════════════════════════════════════════════════════
#  OPTION 2 : Webcam en temps réel (Colab)
# ══════════════════════════════════════════════════════════
from IPython.display import display, Javascript, HTML
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import PIL.Image, io

def capture_webcam():
    """Capture une photo depuis la webcam Colab."""
    js = Javascript('''
    async function capture() {
        const div = document.createElement('div');
        const video = document.createElement('video');
        video.style.display = 'block';
        const stream = await navigator.mediaDevices.getUserMedia({video: true});
        document.body.appendChild(div);
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();

        // Attendre 2s pour que la caméra s'ajuste
        await new Promise(resolve => setTimeout(resolve, 2000));

        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);

        stream.getTracks().forEach(track => track.stop());
        div.remove();
        return canvas.toDataURL('image/jpeg', 0.8);
    }
    ''')
    display(js)
    data = eval_js('capture()')
    binary = b64decode(data.split(',')[1])
    img_array = np.frombuffer(binary, dtype=np.uint8)
    return cv2.imdecode(img_array, cv2.IMREAD_COLOR)

print("📸 Webcam Colab — Capture et analyse")
print("   Captures continues toutes les 2 secondes...\n")

n_captures = 5  # Nombre de captures
for i in range(n_captures):
    try:
        frame = capture_webcam()
        timestamp = time.time()

        annotated = process_frame(frame, model, face_app, person_mgr, tracker, timestamp)

        out_path = f"webcam_capture_{i}.jpg"
        cv2.imwrite(out_path, annotated)
        from IPython.display import Image as IPImage
        display(IPImage(out_path, width=500))

        print(f"  Capture {i+1}/{n_captures} — Personnes actives : {len(tracker.active)}")
        time.sleep(1)
    except Exception as e:
        print(f"  ⚠️ Webcam indisponible : {e}")
        print("  → Utilisez l'option vidéo uploadée (cellule précédente)")
        break

# Rapport
print(tracker.get_report())

## 7️⃣ Test du Modèle — Chargez vos données !

**3 options pour tester :**
1. **Upload d'images** (jpg, png) → détection + reconnaissance instantanée
2. **Upload d'une vidéo** (mp4, avi) → pipeline complet avec tracking
3. **URL d'image** → téléchargement et analyse directe

> Le modèle détecte les personnes, reconnaît les visages enregistrés et affiche le temps de présence.

In [ ]:
# ══════════════════════════════════════════════════════════
#  TEST 1 : Upload d'images
# ══════════════════════════════════════════════════════════
from google.colab import files as colab_files
from IPython.display import display, Image as IPImage
import matplotlib.pyplot as plt

print("📤 Uploadez une ou plusieurs images (jpg, png) :")
print("   → Le modèle va détecter les personnes et tenter de les identifier\n")

uploaded = colab_files.upload()

if uploaded:
    fig_cols = min(len(uploaded), 4)
    fig_rows = (len(uploaded) + fig_cols - 1) // fig_cols
    fig, axes = plt.subplots(fig_rows, fig_cols, figsize=(6*fig_cols, 6*fig_rows))
    if len(uploaded) == 1:
        axes = [axes]
    else:
        axes = axes.flat if hasattr(axes, 'flat') else [axes]

    for idx, (filename, content) in enumerate(uploaded.items()):
        print(f"\n{'─'*50}")
        print(f"  🔍 Analyse : {filename}")
        print(f"{'─'*50}")

        img = cv2.imread(filename)
        if img is None:
            print(f"  ❌ Impossible de lire {filename}")
            continue

        timestamp = time.time()

        # Pipeline complet
        annotated = process_frame(img, model, face_app, person_mgr, tracker, timestamp)

        # Détails
        results = model.predict(img, conf=0.3, classes=[0], verbose=False)[0]
        n_persons = len(results.boxes)
        faces = face_app.get(img)
        n_faces = len(faces)

        print(f"  👥 Personnes détectées : {n_persons}")
        print(f"  😀 Visages détectés   : {n_faces}")

        # Identifier chaque visage
        for i, face in enumerate(faces):
            name, sim = person_mgr.identify(face.embedding)
            if name:
                print(f"  ✅ Visage {i+1} : {name} (similarité {sim:.0%})")
            else:
                print(f"  ❓ Visage {i+1} : Inconnu (meilleur score {sim:.0%})")

        # Afficher
        out_path = f"test_result_{idx}.jpg"
        cv2.imwrite(out_path, annotated)

        if idx < len(list(axes)):
            ax = list(axes)[idx]
            ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
            ax.set_title(f"{filename}\n{n_persons} pers. | {n_faces} visages", fontsize=10)
            ax.axis("off")

    # Cacher les axes vides
    for j in range(idx + 1, len(list(axes))):
        list(axes)[j].axis("off")

    plt.suptitle("Surveillance-IA — Résultats de Détection & Reconnaissance",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("⏭️ Aucune image uploadée")

In [ ]:
# ══════════════════════════════════════════════════════════
#  TEST 2 : Upload d'une vidéo
# ══════════════════════════════════════════════════════════
from google.colab import files as colab_files
from IPython.display import HTML

print("🎬 Uploadez une vidéo (mp4, avi, mov) :")
print("   → Pipeline complet : détection + reconnaissance + tracking")
print("   → Rapport de présence généré à la fin\n")

uploaded_video = colab_files.upload()

if uploaded_video:
    video_name = list(uploaded_video.keys())[0]
    print(f"\n{'═'*60}")
    print(f"  ▶ Traitement de : {video_name}")
    print(f"{'═'*60}\n")

    # Nouveau tracker pour cette vidéo
    video_tracker = PresenceTracker()

    output_path = f"result_{video_name}"
    result = process_video(
        video_path=video_name,
        output_path=output_path,
        model=model,
        face_app=face_app,
        person_mgr=person_mgr,
        tracker_override=video_tracker,
        max_frames=500
    )

    # Le rapport est déjà affiché par process_video()
    print(f"\n  📁 Vidéo annotée sauvegardée : {output_path}")

    # Télécharger le résultat
    try:
        colab_files.download(output_path)
        print("  ✅ Téléchargement lancé !")
    except:
        print("  💡 Utilisez le panneau Fichiers à gauche pour télécharger")
else:
    print("⏭️ Aucune vidéo uploadée")

In [ ]:
# ══════════════════════════════════════════════════════════
#  TEST 3 : Analyse d'image depuis une URL
# ══════════════════════════════════════════════════════════
import urllib.request

url = input("🔗 Entrez l'URL d'une image (ou appuyez sur Entrée pour passer) : ").strip()

if url:
    print(f"\n  ⬇ Téléchargement depuis : {url[:80]}...")

    try:
        img_path = "url_image_test.jpg"
        urllib.request.urlretrieve(url, img_path)

        img = cv2.imread(img_path)
        if img is None:
            raise ValueError("Impossible de décoder l'image")

        timestamp = time.time()

        # Pipeline complet
        annotated = process_frame(img, model, face_app, person_mgr, tracker, timestamp)

        # Détails
        results = model.predict(img, conf=0.3, classes=[0], verbose=False)[0]
        n_persons = len(results.boxes)
        faces = face_app.get(img)
        n_faces = len(faces)

        print(f"\n{'─'*50}")
        print(f"  🔍 Résultat de l'analyse")
        print(f"{'─'*50}")
        print(f"  👥 Personnes détectées : {n_persons}")
        print(f"  😀 Visages détectés   : {n_faces}")

        for i, face in enumerate(faces):
            name, sim = person_mgr.identify(face.embedding)
            if name:
                print(f"  ✅ Visage {i+1} : {name} (similarité {sim:.0%})")
            else:
                print(f"  ❓ Visage {i+1} : Inconnu (meilleur score {sim:.0%})")

        # Afficher
        cv2.imwrite("url_result.jpg", annotated)
        plt.figure(figsize=(12, 8))
        plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        plt.title(f"Résultat — {n_persons} personnes | {n_faces} visages", fontsize=14)
        plt.axis("off")
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"  ❌ Erreur : {e}")
else:
    print("⏭️ Test par URL passé")

## 8️⃣ Sauvegarde — Google Drive + Export

Sauvegardez vos modèles et données pour réutilisation :
- **PersonManager** (base de visages enregistrés)
- **Vidéos annotées**
- **Rapports de présence**

In [ ]:
# ══════════════════════════════════════════════════════════
#  SAUVEGARDE — Google Drive + Export local
# ══════════════════════════════════════════════════════════
from google.colab import drive
import shutil, json, glob

# Monter Google Drive
drive.mount('/content/drive')

# Dossier de sauvegarde
save_dir = "/content/drive/MyDrive/Surveillance_IA"
os.makedirs(save_dir, exist_ok=True)
os.makedirs(os.path.join(save_dir, "models"), exist_ok=True)

# ═══════════════════════════════════════════════════
#  1) EXPORT DES MODÈLES
# ═══════════════════════════════════════════════════

# a) Copier le modèle YOLO original (.pt)
yolo_src = model.ckpt_path if hasattr(model, 'ckpt_path') else "yolov8n.pt"
shutil.copy(yolo_src, os.path.join(save_dir, "models", "yolov8n.pt"))
print(f"✅ YOLOv8n (.pt) copié → models/yolov8n.pt")

# b) Export ONNX (déploiement sans PyTorch — CPU/Edge/Web)
print("\n📦 Export ONNX en cours...")
onnx_path = model.export(format="onnx", imgsz=640, simplify=True)
shutil.copy(onnx_path, os.path.join(save_dir, "models", "yolov8n.onnx"))
print(f"✅ YOLOv8n (.onnx) exporté → models/yolov8n.onnx")

# c) Export TorchScript (inférence PyTorch optimisée)
print("\n📦 Export TorchScript en cours...")
ts_path = model.export(format="torchscript", imgsz=640)
shutil.copy(ts_path, os.path.join(save_dir, "models", "yolov8n.torchscript"))
print(f"✅ YOLOv8n (.torchscript) exporté → models/yolov8n.torchscript")

print(f"\n📊 Modèles exportés :")
for f in os.listdir(os.path.join(save_dir, "models")):
    size_mb = os.path.getsize(os.path.join(save_dir, "models", f)) / 1e6
    print(f"   {f:.<40} {size_mb:.1f} MB")

# ═══════════════════════════════════════════════════
#  2) SAUVEGARDER LA BASE DE PERSONNES
# ═══════════════════════════════════════════════════
person_mgr.save(os.path.join(save_dir, "person_database.json"))
print(f"\n✅ Base de personnes sauvegardée ({len(person_mgr.persons)} personnes)")

# ═══════════════════════════════════════════════════
#  3) COPIER LES RÉSULTATS DE TEST
# ═══════════════════════════════════════════════════
results_copied = 0
for f in glob.glob("test_result_*.jpg") + glob.glob("result_*") + glob.glob("url_result.jpg"):
    shutil.copy(f, save_dir)
    results_copied += 1
    print(f"  📁 Copié : {f}")
if results_copied == 0:
    print("  (aucun résultat de test à copier)")

# ═══════════════════════════════════════════════════
#  4) EXPORTER LE RAPPORT DE PRÉSENCE
# ═══════════════════════════════════════════════════
report_str = tracker.get_report()
if report_str:
    report_path = os.path.join(save_dir, "rapport_presence.txt")
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(report_str)
    print(f"\n✅ Rapport de présence exporté → rapport_presence.txt")

# ═══════════════════════════════════════════════════
#  RÉSUMÉ
# ═══════════════════════════════════════════════════
print(f"\n{'═'*60}")
print(f"  📂 TOUT SAUVEGARDÉ DANS : {save_dir}")
print(f"{'═'*60}")
print(f"  models/yolov8n.pt          → Détection (PyTorch)")
print(f"  models/yolov8n.onnx        → Déploiement CPU/Edge/Web")
print(f"  models/yolov8n.torchscript → Inférence optimisée")
print(f"  person_database.json       → Base de visages")
print(f"  rapport_presence.txt       → Rapport de suivi")
print(f"{'═'*60}")

## 9️⃣ Rapport de Performance — Benchmark

Mesure la vitesse de chaque composant du pipeline pour vérifier si le traitement temps réel est possible.

In [ ]:
# ══════════════════════════════════════════════════════════
#  BENCHMARK — Performance du pipeline
# ══════════════════════════════════════════════════════════
import time

# Image de test
test_img = cv2.imread("test_person.jpg")

# Warmup
for _ in range(10):
    model.predict(test_img, conf=0.3, classes=[0], verbose=False)
    face_app.get(test_img)

# Mesure YOLO
t0 = time.time()
for _ in range(100):
    model.predict(test_img, conf=0.3, classes=[0], verbose=False)
yolo_ms = (time.time() - t0) / 100 * 1000

# Mesure InsightFace
t0 = time.time()
for _ in range(100):
    face_app.get(test_img)
face_ms = (time.time() - t0) / 100 * 1000

total_ms = yolo_ms + face_ms
fps = 1000 / total_ms

print(f"{'='*55}")
print(f"  {'BENCHMARK PERFORMANCE':^51}")
print(f"{'='*55}")
print(f"  {'YOLOv8n (détection)':.<30} {yolo_ms:.1f} ms")
print(f"  {'InsightFace (reconnaissance)':.<30} {face_ms:.1f} ms")
print(f"  {'Pipeline complet':.<30} {total_ms:.1f} ms")
print(f"  {'FPS estimé':.<30} {fps:.1f}")
print(f"{'='*55}")

if fps >= 25:
    print("  ✅ TEMPS RÉEL OK (≥25 FPS)")
elif fps >= 15:
    print("  🟡 Acceptable pour surveillance (~15 FPS)")
else:
    print("  ⚠️ Lent — considérer YOLOv8n + skip frames")

print(f"\n📊 Personnes enregistrées : {len(person_mgr.persons)}")
print(f"📊 Personnes actuellement suivies : {len(tracker.active)}")

---

## ✅ Récapitulatif Final

### Pipeline Surveillance-IA
| Étape | Technologie | Temps |
|-------|------------|-------|
| Détection | YOLOv8n pré-entraîné | **0s** (pas de training) |
| Reconnaissance | InsightFace buffalo_l | **~2min** (téléchargement modèle) |
| Enregistrement | PersonManager (JSON) | **~10s** par personne |
| Tracking | PresenceTracker | Temps réel |

### Fichiers exportés sur Google Drive
| Fichier | Usage |
|---------|-------|
| `yolov8n.pt` | Détection de personnes |
| `yolov8n.onnx` | Déploiement CPU (sans PyTorch) |
| `person_database.json` | Base des personnes enregistrées |

### Déploiement local

Téléchargez les fichiers et placez-les dans votre projet :

```
surveillance/
├── models/
│   └── finetuned/
│       └── yolov8n.pt
├── data/
│   └── person_database.json
└── src/
    └── tracker.py
```

```bash
# Tracking temps réel sur webcam
python -m src.tracker --model models/finetuned/yolov8n.pt --source 0 --show

# API REST
uvicorn api.main:app --host 0.0.0.0 --port 8000

# Docker
docker compose up -d
```

---

**Auteur** : BAWANA Théodore — [theo.portefolio.io](https://theo.portefolio.io) — [GitHub](https://github.com/theobawana) — Projet SAHELYS